In [ ]:
!pip -q install torch torchaudio pandas scikit-learn tqdm

import os, zipfile, glob, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import timm

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
results_dict={}

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device != "cuda":
    print("WARNING: switch Colab runtime to GPU for best speed.")

Device: cuda


In [ ]:
TRAIN_ZIP = "/content/Train-20260303T232131Z-3-001.zip"
VALID_ZIP = "/content/Valid-20260303T232008Z-3-001.zip"
TEST_ZIP  = "/content/Test-20260303T232005Z-3-001.zip"

OUT_DIR = "/content/keystroke_dataset_attn"
os.makedirs(OUT_DIR, exist_ok=True)

def unzip(zip_path, out_dir):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

print("Unzipping...")
unzip(TRAIN_ZIP, OUT_DIR)
unzip(VALID_ZIP, OUT_DIR)
unzip(TEST_ZIP,  OUT_DIR)

def find_split_dir(root, split_name):
    cands = glob.glob(os.path.join(root, "**", split_name), recursive=True)
    if not cands:
        raise FileNotFoundError(f"Could not find split folder '{split_name}' under {root}")
    return cands[0]

train_dir = find_split_dir(OUT_DIR, "Train")
valid_dir = find_split_dir(OUT_DIR, "Valid")
test_dir  = find_split_dir(OUT_DIR, "Test")

def load_labels(split_dir):
    csvs = glob.glob(os.path.join(split_dir, "**", "labels.csv"), recursive=True)
    if not csvs:
        raise FileNotFoundError(f"No labels.csv found under {split_dir}")
    df = pd.read_csv(csvs[0])
    if "filename" not in df.columns or "key" not in df.columns:
        raise ValueError(f"labels.csv must contain filename,key. Found: {df.columns.tolist()}")
    return df

def find_audio_root(split_dir):
    cands = glob.glob(os.path.join(split_dir, "**", "audio_data"), recursive=True)
    if cands:
        return cands[0]
    wavs = glob.glob(os.path.join(split_dir, "**", "*.wav"), recursive=True)
    if not wavs:
        raise FileNotFoundError(f"No .wav found under {split_dir}")
    return os.path.dirname(wavs[0])

train_df = load_labels(train_dir)
valid_df = load_labels(valid_dir)
test_df  = load_labels(test_dir)

train_audio_root = find_audio_root(train_dir)
valid_audio_root = find_audio_root(valid_dir)
test_audio_root  = find_audio_root(test_dir)

class_names = sorted(train_df["key"].unique().tolist())
label2id = {k:i for i,k in enumerate(class_names)}
id2label = {i:k for k,i in label2id.items()}
num_classes = len(class_names)
print("Num classes:", num_classes, class_names)

def resolve_path(audio_root, fname):
    p = os.path.join(audio_root, fname)
    if os.path.exists(p):
        return p
    matches = glob.glob(os.path.join(os.path.dirname(audio_root), "**", fname), recursive=True)
    if not matches:
        raise FileNotFoundError(f"Missing file: {fname}")
    return matches[0]

Unzipping...
Num classes: 27 ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 'space', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [ ]:
TARGET_SR = 16000
MAX_SECONDS = 0.38
MAX_LEN = int(TARGET_SR * MAX_SECONDS)

# Keep small for speed
N_FFT = 512
HOP = 160
N_MELS = 64
FMIN, FMAX = 50, 8000

mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP,
    n_mels=N_MELS,
    f_min=FMIN,
    f_max=FMAX,
    power=2.0
)

def wav_to_logmel(wav_1d: torch.Tensor) -> torch.Tensor:
    # wav_1d: (T,)
    x = wav_1d.unsqueeze(0)                 # (1,T)
    m = mel(x).squeeze(0)                   # (n_mels, frames)
    m = torch.log(m + 1e-6)
    # normalize
    m = (m - m.mean()) / (m.std() + 1e-6)
    return m                                # (n_mels, frames)

class LogMelDataset(torch.utils.data.Dataset):

    def __init__(self, df, audio_root, cache=True):
        self.df = df.reset_index(drop=True)
        self.audio_root = audio_root
        self.cache = {} if cache else None

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row["filename"]
        y = label2id[row["key"]]

        if self.cache is not None and fname in self.cache:
            return self.cache[fname], torch.tensor(y, dtype=torch.long)

        path = resolve_path(self.audio_root, fname)
        wav, sr = torchaudio.load(path)
        wav = wav.mean(dim=0)

        if sr != TARGET_SR:
            wav = torchaudio.functional.resample(wav, sr, TARGET_SR)

        if wav.numel() < MAX_LEN:
            wav = F.pad(wav, (0, MAX_LEN - wav.numel()))
        else:
            wav = wav[:MAX_LEN]

        feat = wav_to_logmel(wav)           # (M, T)

        if self.cache is not None:
            self.cache[fname] = feat

        return feat, torch.tensor(y, dtype=torch.long)

train_ds = LogMelDataset(train_df, train_audio_root, cache=True)
valid_ds = LogMelDataset(valid_df, valid_audio_root, cache=True)
test_ds  = LogMelDataset(test_df,  test_audio_root,  cache=True)

def collate(batch):
    # All clips are same length after pad/trim => same frames
    X, y = zip(*batch)
    X = torch.stack(X, dim=0)  # (B, M, T)
    y = torch.stack(y, dim=0)  # (B,)
    return X, y

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True, collate_fn=collate)
valid_loader = torch.utils.data.DataLoader(valid_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=128, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate)

In [ ]:
class CNN_Attn(nn.Module):

    def __init__(self, n_mels=64, n_classes=27, d_model=128, n_heads=4, dropout=0.1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.MaxPool2d((2, 2)),      # (M/2, T/2)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.MaxPool2d((2, 2)),      # (M/4, T/4)

            nn.Conv2d(64, d_model, kernel_size=3, padding=1),
            nn.BatchNorm2d(d_model),
            nn.GELU(),
        )

        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, x):
        # x: (B, M, T)
        x = x.unsqueeze(1)            # (B,1,M,T)
        z = self.stem(x)              # (B,d_model,M',T')

        # Pool over frequency (M') to get tokens over time
        z = z.mean(dim=2)             # (B,d_model,T')
        z = z.transpose(1, 2)         # (B,T',d_model) tokens

        # Self-attention
        attn_out, _ = self.attn(z, z, z, need_weights=False)
        z = z + attn_out
        z = z + self.ffn(z)

        # Global average pool over time tokens
        z = z.mean(dim=1)             # (B,d_model)
        return self.head(z)           # (B,n_classes)

model = CNN_Attn(n_mels=N_MELS, n_classes=num_classes, d_model=128, n_heads=4, dropout=0.1).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

scaler = torch.amp.GradScaler('cuda',enabled=(device == "cuda"))

@torch.no_grad()
def evaluate(loader):
    model.eval()
    ys, ps = [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            logits = model(X)
        pred = logits.argmax(dim=1)
        ys.append(y.cpu().numpy())
        ps.append(pred.cpu().numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps)
    return accuracy_score(ys, ps), ys, ps

Training with minimum epochs

In [ ]:
best_val = 0.0
best_path = "/content/best_cnn_attn_keystrokes.pt"
patience = 5
bad = 0
EPOCHS = 20

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0

    for X, y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            logits = model(X)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(optimizer)
        scaler.update()

        running += loss.item() * y.size(0)

    train_loss = running / len(train_loader.dataset)
    val_acc, _, _ = evaluate(valid_loader)
    scheduler.step(val_acc)

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}  val_acc={val_acc:.4f}")

    if val_acc > best_val:
        best_val = val_acc
        bad = 0
        torch.save({
            "model_state": model.state_dict(),
            "class_names": class_names,
            "cfg": {"N_FFT": N_FFT, "HOP": HOP, "N_MELS": N_MELS, "MAX_SECONDS": MAX_SECONDS, "SR": TARGET_SR}
        }, best_path)
        print("Saved best:", best_path)
    else:
        bad += 1
        if bad >= patience:
            print("Early stopping.")
            break

print("Best val acc:", best_val)

Epoch 1/20:   0%|          | 0/80 [00:00<?, ?it/s]/tmp/ipykernel_3915/1320071974.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):
Epoch 1/20: 100%|██████████| 80/80 [00:30<00:00,  2.60it/s]
/tmp/ipykernel_3915/933109945.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Epoch 1: train_loss=2.3936  val_acc=0.3578
Saved best: /content/best_cnn_attn_keystrokes.pt


Epoch 2/20:   0%|          | 0/80 [00:00<?, ?it/s]/tmp/ipykernel_3915/1320071974.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):
Epoch 2/20: 100%|██████████| 80/80 [00:30<00:00,  2.63it/s]
/tmp/ipykernel_3915/933109945.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Epoch 2: train_loss=1.2976  val_acc=0.6039
Saved best: /content/best_cnn_attn_keystrokes.pt


Epoch 3/20:   0%|          | 0/80 [00:00<?, ?it/s]/tmp/ipykernel_3915/1320071974.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):
Epoch 3/20: 100%|██████████| 80/80 [00:28<00:00,  2.78it/s]
/tmp/ipykernel_3915/933109945.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


...

Epoch 18: train_loss=0.0110  val_acc=0.9461


Epoch 19/20:   0%|          | 0/80 [00:00<?, ?it/s]/tmp/ipykernel_3915/1320071974.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):
Epoch 19/20: 100%|██████████| 80/80 [00:29<00:00,  2.71it/s]
/tmp/ipykernel_3915/933109945.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Epoch 19: train_loss=0.0112  val_acc=0.9539


Epoch 20/20:   0%|          | 0/80 [00:00<?, ?it/s]/tmp/ipykernel_3915/1320071974.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):
Epoch 20/20: 100%|██████████| 80/80 [00:29<00:00,  2.72it/s]
/tmp/ipykernel_3915/933109945.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):


Epoch 20: train_loss=0.0099  val_acc=0.9672
Best val acc: 0.9671875


Testing for Clean Accuracy & Confusion Matrix

In [ ]:
ck = torch.load(best_path, map_location=device)
model.load_state_dict(ck["model_state"])

test_acc, y_true, y_pred = evaluate(test_loader)
print("\nTEST accuracy:", test_acc)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
cm_path = "/content/cnn_attn_confusion_matrix.csv"
pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_path)
print("\nSaved confusion matrix:", cm_path)
print("Saved best model:", best_path)

/tmp/ipykernel_3915/933109945.py:73: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):



TEST accuracy: 0.96

Classification report:
              precision    recall  f1-score   support

           a     0.9828    0.9661    0.9744        59
           b     0.9355    0.9831    0.9587        59
           c     0.9828    0.9661    0.9744        59
           d     0.9667    0.9831    0.9748        59
           e     1.0000    0.9831    0.9915        59
           f     0.9821    0.9322    0.9565        59
           g     0.9492    0.9492    0.9492        59
           h     1.0000    0.9322    0.9649        59
           i     0.9833    0.9833    0.9833        60
           j     0.9500    0.9661    0.9580        59
           k     0.9344    0.9661    0.9500        59
           l     1.0000    1.0000    1.0000        59
           m     0.8529    0.9831    0.9134        59
           n     0.9219    1.0000    0.9593        59
           o     1.0000    0.9831    0.9915        59
           p     0.9667    0.9831    0.9748        59
           q     0.9831    0.9667   

Adversarial Attacks (FGSM, BIM, PGD and DFT)

In [ ]:
!pip install adversarial-robustness-toolbox -q

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, BasicIterativeMethod, ProjectedGradientDescent

print("Collecting test features...")

model.eval()

X_test_list = []
y_test_list = []

for x, y in test_loader:
    X_test_list.append(x.cpu().numpy())
    y_test_list.append(y.cpu().numpy())

X_test = np.concatenate(X_test_list).astype(np.float32)
y_test = np.concatenate(y_test_list)

print("Test shape:", X_test.shape)

T_FRAMES = X_test.shape[2]

device_type = "gpu" if device == "cuda" else "cpu"

classifier = PyTorchClassifier(
    model=model,
    loss=criterion,
    optimizer=optimizer,
    input_shape=(N_MELS, T_FRAMES),
    nb_classes=num_classes,
    clip_values=(X_test.min(), X_test.max()),
    device_type=device_type
)

# RESULTS STORAGE
results = []

results_dict = {
    "model_name": "CNN + Self-Attention",
    "clean_accuracy": None,
    "attacks": {}
}

# EVALUATION FUNCTION
def evaluate_attack_per_class(model_name, classifier, X_clean, y_true, attack, attack_name, epsilon):

    print(f"\nRunning {attack_name} ε={epsilon}")

    X_adv = attack.generate(x=X_clean)

    y_pred_clean = np.argmax(classifier.predict(X_clean), axis=1)
    y_pred_adv = np.argmax(classifier.predict(X_adv), axis=1)

    overall_clean = accuracy_score(y_true, y_pred_clean)
    overall_adv = accuracy_score(y_true, y_pred_adv)

    print(f"\nOverall Clean Accuracy : {overall_clean*100:.2f}%")
    print(f"Overall Adv Accuracy   : {overall_adv*100:.2f}%")

    print("\nPer-Keystroke Results")

    for c in range(num_classes):

        idx = (y_true == c)

        if np.sum(idx) == 0:
            continue

        clean_correct = np.sum(y_pred_clean[idx] == c)
        adv_correct = np.sum(y_pred_adv[idx] == c)

        clean_acc = clean_correct / np.sum(idx)
        adv_acc = adv_correct / np.sum(idx)

        asr = 1 - adv_acc

        key_name = class_names[c]

        print(f"{key_name:>6} | Clean {clean_acc*100:6.2f}% | Adv {adv_acc*100:6.2f}% | ASR {asr*100:6.2f}%")

        results.append({
            "model": model_name,
            "attack": attack_name,
            "epsilon": epsilon,
            "key": key_name,
            "clean_acc": round(clean_acc*100, 2),
            "adv_acc": round(adv_acc*100, 2),
            "attack_success_rate": round(asr*100, 2)
        })

    return overall_clean, overall_adv

model_name = "CNN + Self-Attention"
epsilons = [0.005, 0.01, 0.02, 0.05]

for eps in epsilons:

    print("\n" + "="*70)
    print("EPSILON:", eps)
    print("="*70)

    # FGSM
    fgsm = FastGradientMethod(estimator=classifier, eps=eps)
    clean_fgsm, adv_fgsm = evaluate_attack_per_class(
        model_name, classifier, X_test, y_test, fgsm, "FGSM", eps
    )

    # Store clean accuracy only once
    if results_dict["clean_accuracy"] is None:
        results_dict["clean_accuracy"] = clean_fgsm

    results_dict["attacks"].setdefault("FGSM", {})[eps] = adv_fgsm


    # BIM
    bim = BasicIterativeMethod(
        estimator=classifier,
        eps=eps,
        eps_step=eps/10,
        max_iter=20
    )
    clean_bim, adv_bim = evaluate_attack_per_class(
        model_name, classifier, X_test, y_test, bim, "BIM", eps
    )

    results_dict["attacks"].setdefault("BIM", {})[eps] = adv_bim


    # PGD
    pgd = ProjectedGradientDescent(
        estimator=classifier,
        eps=eps,
        eps_step=eps/10,
        max_iter=40,
        num_random_init=1
    )
    clean_pgd, adv_pgd = evaluate_attack_per_class(
        model_name, classifier, X_test, y_test, pgd, "PGD", eps
    )

    results_dict["attacks"].setdefault("PGD", {})[eps] = adv_pgd


# SAVE PER-CLASS RESULTS
os.makedirs("/content/results", exist_ok=True)

df = pd.DataFrame(results)

save_path = "/content/results/keystroke_attack_per_class_results.csv"
df.to_csv(save_path, index=False)

print("\nAll attacks completed")
print("Results saved to:", save_path)

print("\nresults_dict:")
print(results_dict)

print("\nPreview:")
print(df.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.8 MB/s eta 0:00:00
Test shape: (1600, 64, 39)

EPSILON: 0.005

Running FGSM ε=0.005

Overall Clean Accuracy : 96.00%
Overall Adv Accuracy   : 94.19%

Per-Keystroke Results
     a | Clean  96.61% | Adv  93.22% | ASR   6.78%
     b | Clean  98.31% | Adv  94.92% | ASR   5.08%
     c | Clean  96.61% | Adv 100.00% | ASR   0.00%
     d | Clean  98.31% | Adv  91.53% | ASR   8.47%
     e | Clean  98.31% | Adv  98.31% | ASR   1.69%
     f | Clean  93.22% | Adv  89.83% | ASR  10.17%
     g | Clean  94.92% | Adv  93.22% | ASR   6.78%
     h | Clean  93.22% | Adv  91.53% | ASR   8.47%
     i | Clean  98.33% | Adv  98.33% | ASR   1.67%
     j | Clean  96.61% | Adv  93.22% | ASR   6.78%
     k | Clean  96.61% | Adv  94.92% | ASR   5.08%
     l | Clean 100.00% | Adv 100.00% | ASR   0.00%
     m | Clean  98.31% | Adv  96.61% | ASR   3.39%
     n | Clean 100.00% | Adv  96.61% | ASR   3.39%
     o | Clean  98.31% | Adv  94.92% | ASR   5.08%
     

PGD - Batches:   0%|          | 0/50 [00:00<?, ?it/s]


Overall Clean Accuracy : 96.00%
Overall Adv Accuracy   : 94.00%

Per-Keystroke Results
     a | Clean  96.61% | Adv  93.22% | ASR   6.78%
     b | Clean  98.31% | Adv  94.92% | ASR   5.08%
     c | Clean  96.61% | Adv 100.00% | ASR   0.00%
     d | Clean  98.31% | Adv  91.53% | ASR   8.47%
     e | Clean  98.31% | Adv  98.31% | ASR   1.69%
     f | Clean  93.22% | Adv  89.83% | ASR  10.17%
     g | Clean  94.92% | Adv  93.22% | ASR   6.78%
     h | Clean  93.22% | Adv  91.53% | ASR   8.47%
     i | Clean  98.33% | Adv  98.33% | ASR   1.67%
     j | Clean  96.61% | Adv  93.22% | ASR   6.78%
     k | Clean  96.61% | Adv  93.22% | ASR   6.78%
     l | Clean 100.00% | Adv 100.00% | ASR   0.00%
     m | Clean  98.31% | Adv  96.61% | ASR   3.39%
     n | Clean 100.00% | Adv  96.61% | ASR   3.39%
     o | Clean  98.31% | Adv  94.92% | ASR   5.08%
     p | Clean  98.31% | Adv 100.00% | ASR   0.00%
     q | Clean  96.67% | Adv  96.67% | ASR   3.33%
     r | Clean 100.00% | Adv 100.00% | ASR   

PGD - Batches:   0%|          | 0/50 [00:00<?, ?it/s]


Overall Clean Accuracy : 96.00%
Overall Adv Accuracy   : 94.00%

Per-Keystroke Results
     a | Clean  96.61% | Adv  93.22% | ASR   6.78%
     b | Clean  98.31% | Adv  94.92% | ASR   5.08%
     c | Clean  96.61% | Adv 100.00% | ASR   0.00%
     d | Clean  98.31% | Adv  91.53% | ASR   8.47%
     e | Clean  98.31% | Adv  98.31% | ASR   1.69%
     f | Clean  93.22% | Adv  89.83% | ASR  10.17%
     g | Clean  94.92% | Adv  93.22% | ASR   6.78%
     h | Clean  93.22% | Adv  91.53% | ASR   8.47%
     i | Clean  98.33% | Adv  98.33% | ASR   1.67%
     j | Clean  96.61% | Adv  93.22% | ASR   6.78%
     k | Clean  96.61% | Adv  93.22% | ASR   6.78%
     l | Clean 100.00% | Adv 100.00% | ASR   0.00%
     m | Clean  98.31% | Adv  96.61% | ASR   3.39%
     n | Clean 100.00% | Adv  96.61% | ASR   3.39%
     o | Clean  98.31% | Adv  94.92% | ASR   5.08%
     p | Clean  98.31% | Adv 100.00% | ASR   0.00%
     q | Clean  96.67% | Adv  96.67% | ASR   3.33%
     r | Clean 100.00% | Adv 100.00% | ASR   

...


Overall Clean Accuracy : 96.00%
Overall Adv Accuracy   : 59.31%

Per-Keystroke Results
     a | Clean  96.61% | Adv  49.15% | ASR  50.85%
     b | Clean  98.31% | Adv  55.93% | ASR  44.07%
     c | Clean  96.61% | Adv  76.27% | ASR  23.73%
     d | Clean  98.31% | Adv  52.54% | ASR  47.46%
     e | Clean  98.31% | Adv  55.93% | ASR  44.07%
     f | Clean  93.22% | Adv  59.32% | ASR  40.68%
     g | Clean  94.92% | Adv  66.10% | ASR  33.90%
     h | Clean  93.22% | Adv  61.02% | ASR  38.98%
     i | Clean  98.33% | Adv  76.67% | ASR  23.33%
     j | Clean  96.61% | Adv  55.93% | ASR  44.07%
     k | Clean  96.61% | Adv  66.10% | ASR  33.90%
     l | Clean 100.00% | Adv  81.36% | ASR  18.64%
     m | Clean  98.31% | Adv  77.97% | ASR  22.03%
     n | Clean 100.00% | Adv  59.32% | ASR  40.68%
     o | Clean  98.31% | Adv  55.93% | ASR  44.07%
     p | Clean  98.31% | Adv  76.27% | ASR  23.73%
     q | Clean  96.67% | Adv  56.67% | ASR  43.33%
     r | Clean 100.00% | Adv  86.44% | ASR  1

PGD - Batches:   0%|          | 0/50 [00:00<?, ?it/s]


Overall Clean Accuracy : 96.00%
Overall Adv Accuracy   : 3.25%

Per-Keystroke Results
     a | Clean  96.61% | Adv   1.69% | ASR  98.31%
     b | Clean  98.31% | Adv   1.69% | ASR  98.31%
     c | Clean  96.61% | Adv   5.08% | ASR  94.92%
     d | Clean  98.31% | Adv   1.69% | ASR  98.31%
     e | Clean  98.31% | Adv   0.00% | ASR 100.00%
     f | Clean  93.22% | Adv   1.69% | ASR  98.31%
     g | Clean  94.92% | Adv   1.69% | ASR  98.31%
     h | Clean  93.22% | Adv   5.08% | ASR  94.92%
     i | Clean  98.33% | Adv   5.00% | ASR  95.00%
     j | Clean  96.61% | Adv   1.69% | ASR  98.31%
     k | Clean  96.61% | Adv   1.69% | ASR  98.31%
     l | Clean 100.00% | Adv   8.47% | ASR  91.53%
     m | Clean  98.31% | Adv   0.00% | ASR 100.00%
     n | Clean 100.00% | Adv   0.00% | ASR 100.00%
     o | Clean  98.31% | Adv   1.69% | ASR  98.31%
     p | Clean  98.31% | Adv   3.39% | ASR  96.61%
     q | Clean  96.67% | Adv   1.67% | ASR  98.33%
     r | Clean 100.00% | Adv   0.00% | ASR 100

PGD - Batches:   0%|          | 0/50 [00:00<?, ?it/s]


Overall Clean Accuracy : 96.00%
Overall Adv Accuracy   : 2.94%

Per-Keystroke Results
     a | Clean  96.61% | Adv   3.39% | ASR  96.61%
     b | Clean  98.31% | Adv   1.69% | ASR  98.31%
     c | Clean  96.61% | Adv   3.39% | ASR  96.61%
     d | Clean  98.31% | Adv   1.69% | ASR  98.31%
     e | Clean  98.31% | Adv   0.00% | ASR 100.00%
     f | Clean  93.22% | Adv   3.39% | ASR  96.61%
     g | Clean  94.92% | Adv   1.69% | ASR  98.31%
     h | Clean  93.22% | Adv   5.08% | ASR  94.92%
     i | Clean  98.33% | Adv   0.00% | ASR 100.00%
     j | Clean  96.61% | Adv   3.39% | ASR  96.61%
     k | Clean  96.61% | Adv   1.69% | ASR  98.31%
     l | Clean 100.00% | Adv   6.78% | ASR  93.22%
     m | Clean  98.31% | Adv   0.00% | ASR 100.00%
     n | Clean 100.00% | Adv   0.00% | ASR 100.00%
     o | Clean  98.31% | Adv   1.69% | ASR  98.31%
     p | Clean  98.31% | Adv   3.39% | ASR  96.61%
     q | Clean  96.67% | Adv   1.67% | ASR  98.33%
     r | Clean 100.00% | Adv   0.00% | ASR 100

DFT - Attack

In [ ]:
print("Starting DFT Attack (Spectral Perturbation - Phase + Amplitude)...")

import numpy as np
from sklearn.metrics import accuracy_score

def dft_attack(X, epsilon, perturb_phase=True, perturb_magnitude=True, high_freq_only=False):

    X_adv = []

    for sample in X:
        # sample shape: (channels, timesteps)

        fft = np.fft.fft(sample, axis=1)

        magnitude = np.abs(fft)
        phase     = np.angle(fft)

        n_freqs   = magnitude.shape[1]

        # Build frequency mask
        if high_freq_only:
            mask = np.zeros_like(magnitude)
            k = n_freqs // 4                 # top 25% high-frequency bins
            mask[:, -k:] = 1.0
        else:
            mask = np.ones_like(magnitude)   # perturb all frequencies

        # --- Magnitude perturbation (relative to signal energy) ---
        if perturb_magnitude:
            mag_noise          = epsilon * magnitude * np.random.randn(*magnitude.shape)
            perturbed_mag      = magnitude + mask * mag_noise
            perturbed_mag      = np.clip(perturbed_mag, 0, None)   # magnitude must stay >= 0
        else:
            perturbed_mag      = magnitude

        # --- Phase perturbation (disrupts temporal alignment) ---
        if perturb_phase:
            phase_noise        = epsilon * np.pi * np.random.randn(*phase.shape)
            perturbed_phase    = phase + mask * phase_noise
        else:
            perturbed_phase    = phase

        # --- Reconstruct adversarial signal ---
        perturbed_fft = perturbed_mag * np.exp(1j * perturbed_phase)
        adv           = np.fft.ifft(perturbed_fft, axis=1).real

        # --- Clip to original sample's value range ---
        adv = np.clip(adv, sample.min(), sample.max())

        X_adv.append(adv)

    return np.array(X_adv).astype(np.float32)

# EVALUATION
def evaluate_dft_attack(model_name, classifier, X_clean, y_true, epsilon,
                        perturb_phase=True, perturb_magnitude=True, high_freq_only=False):

    print(f"\nRunning DFT Spectral Attack  ε = {epsilon}"
          f"  | phase={perturb_phase}  mag={perturb_magnitude}  high_freq_only={high_freq_only}")

    X_adv = dft_attack(
        X_clean, epsilon,
        perturb_phase=perturb_phase,
        perturb_magnitude=perturb_magnitude,
        high_freq_only=high_freq_only
    )

    # Diagnostic — confirm perturbation is meaningful
    diff = np.abs(X_adv - X_clean)
    print(f"  Perturbation  mean={diff.mean():.6f}  max={diff.max():.6f}  "
          f"signal_mean={np.abs(X_clean).mean():.6f}")

    y_pred_clean = np.argmax(classifier.predict(X_clean), axis=1)
    y_pred_adv   = np.argmax(classifier.predict(X_adv),   axis=1)

    overall_clean = accuracy_score(y_true, y_pred_clean)
    overall_adv   = accuracy_score(y_true, y_pred_adv)

    print(f"\n  Overall Clean Accuracy : {overall_clean * 100:.2f}%")
    print(f"  Overall Adv   Accuracy : {overall_adv   * 100:.2f}%")
    print(f"  Accuracy Drop          : {(overall_clean - overall_adv) * 100:.2f}%")

    print("\n  Per-Keystroke Results:")

    for c in range(num_classes):

        idx = (y_true == c)

        if np.sum(idx) == 0:
            continue

        clean_acc_class = np.sum(y_pred_clean[idx] == c) / np.sum(idx)
        adv_acc_class   = np.sum(y_pred_adv[idx]   == c) / np.sum(idx)
        asr             = 1 - adv_acc_class
        key_name        = class_names[c]

        print(f"  {key_name:>6} | Clean {clean_acc_class*100:6.2f}%"
              f" | Adv {adv_acc_class*100:6.2f}% | ASR {asr*100:6.2f}%")

        results.append({
            "model":               model_name,
            "attack":              "DFT_Spectral",
            "epsilon":             epsilon,
            "key":                 key_name,
            "clean_acc":           round(clean_acc_class * 100, 2),
            "adv_acc":             round(adv_acc_class   * 100, 2),
            "attack_success_rate": round(asr             * 100, 2)
        })

    return overall_clean, overall_adv

# RUN - Four epsilon values, full-spectrum phase + magnitude
model_name = "CNN + Self-Attention"
epsilons   = [0.005, 0.01, 0.02, 0.05]

for eps in epsilons:

    print("\n" + "="*60)
    print(f"  DFT SPECTRAL ATTACK — EPSILON: {eps}")
    print("="*60)

    clean_dft, adv_dft = evaluate_dft_attack(
        model_name,
        classifier,
        X_test,
        y_test,
        epsilon          = eps,
        perturb_phase    = True,    # phase disrupts temporal alignment
        perturb_magnitude= True,    # relative amplitude noise
        high_freq_only   = False    # perturb full spectrum
    )

    if results_dict["clean_accuracy"] is None:
        results_dict["clean_accuracy"] = clean_dft

    results_dict["attacks"].setdefault("DFT_Spectral", {})[eps] = adv_dft


# SAVE
df        = pd.DataFrame(results)
save_path = "/content/results/final_attack_per_class_results.csv"
df.to_csv(save_path, index=False)

print("\nDFT Spectral Attack Completed.")
print("Results saved to:", save_path)

print("\nUpdated results_dict:")
print(results_dict)

print("\nPreview:")
print(df.head())

Starting DFT Attack (Spectral Perturbation - Phase + Amplitude)...

  DFT SPECTRAL ATTACK — EPSILON: 0.005

Running DFT Spectral Attack  ε = 0.005  | phase=True  mag=True  high_freq_only=False
  Perturbation  mean=0.006821  max=0.090343  signal_mean=0.764063

  Overall Clean Accuracy : 96.00%
  Overall Adv   Accuracy : 95.88%
  Accuracy Drop          : 0.12%

  Per-Keystroke Results:
       a | Clean  96.61% | Adv  96.61% | ASR   3.39%
       b | Clean  98.31% | Adv  98.31% | ASR   1.69%
       c | Clean  96.61% | Adv  96.61% | ASR   3.39%
       d | Clean  98.31% | Adv  98.31% | ASR   1.69%
       e | Clean  98.31% | Adv  98.31% | ASR   1.69%
       f | Clean  93.22% | Adv  93.22% | ASR   6.78%
       g | Clean  94.92% | Adv  94.92% | ASR   5.08%
       h | Clean  93.22% | Adv  93.22% | ASR   6.78%
       i | Clean  98.33% | Adv  98.33% | ASR   1.67%
       j | Clean  96.61% | Adv  96.61% | ASR   3.39%
       k | Clean  96.61% | Adv  96.61% | ASR   3.39%
       l | Clean 100.00% | Adv

In [ ]:
print("Total Results : ", results_dict)

Total Results :  {'model_name': 'CNN + Self-Attention', 'clean_accuracy': 0.96, 'attacks': {'FGSM': {0.005: 0.941875, 0.01: 0.885, 0.02: 0.684375, 0.05: 0.168125}, 'BIM': {0.005: 0.94, 0.01: 0.87375, 0.02: 0.605, 0.05: 0.0325}, 'PGD': {0.005: 0.94, 0.01: 0.87, 0.02: 0.593125, 0.05: 0.029375}, 'DFT_Spectral': {0.005: 0.95875, 0.01: 0.960625, 0.02: 0.958125, 0.05: 0.946875}}}


In [ ]:
import pandas as pd

# 1. Create an empty list to store the flattened rows
summary_rows = []

# Extract top-level model info from your results_dict
model_name = results_dict["model_name"]
# Multiply by 100 to format as a percentage, matching your print statements
clean_acc = results_dict["clean_accuracy"] * 100

# 2. Loop through the attacks and their corresponding epsilon values
for attack_type, eps_data in results_dict["attacks"].items():
    for epsilon, adv_acc in eps_data.items():

        # Convert adversarial accuracy to percentage
        adv_acc_pct = adv_acc * 100

        # Calculate relative accuracy w.r.t clean
        rel_acc = (adv_acc_pct/clean_acc)*100

        # Calculate ASR (100% - Adversary Accuracy%)
        asr = 100.0 - rel_acc

        # Append the row with the exact column names you requested
        summary_rows.append({
            "model_name": model_name,
            "Clean accuracy": round(clean_acc, 2),
            "attack type": attack_type,
            "epsilon": epsilon,
            "adversarial accuracy": round(adv_acc_pct, 2),
            "relative accuracy": round(rel_acc,2),
            "ASR": round(asr, 2)
        })

# 3. Convert to a Pandas DataFrame
df_overall_summary = pd.DataFrame(summary_rows)

# 4. Save to CSV
output_csv_name = "CNN_SelfAttn_overall_attack_summary.csv"
df_overall_summary.to_csv(output_csv_name, index=False)

print(f"✅ Successfully saved to {output_csv_name}\n")
print("Preview of the data:")
display(df_overall_summary)

✅ Successfully saved to CNN_SelfAttn_overall_attack_summary.csv

Preview of the data:


,model_name,Clean accuracy,attack type,epsilon,adversarial accuracy,relative accuracy,ASR
0,CNN + Self-Attention,96.0,FGSM,0.005,94.19,98.11,1.89
1,CNN + Self-Attention,96.0,FGSM,0.010,88.50,92.19,7.81
2,CNN + Self-Attention,96.0,FGSM,0.020,68.44,71.29,28.71
3,CNN + Self-Attention,96.0,FGSM,0.050,16.81,17.51,82.49
4,CNN + Self-Attention,96.0,BIM,0.005,94.00,97.92,2.08
5,CNN + Self-Attention,96.0,BIM,0.010,87.38,91.02,8.98
6,CNN + Self-Attention,96.0,BIM,0.020,60.50,63.02,36.98
7,CNN + Self-Attention,96.0,BIM,0.050,3.25,3.39,96.61
8,CNN + Self-Attention,96.0,PGD,0.005,94.00,97.92,2.08
9,CNN + Self-Attention,96.0,PGD,0.010,87.00,90.62,9.38
